# 小心使用浮點數與 `Decimal`

1. 為什麼浮點數運算會有誤差
2. 如何避免使用浮點數（轉成整數運算）
3. `Decimal` 的正確用法與注意事項
4. `Decimal` 的效能代價


## 1. 浮點數的誤差

電腦用二進位儲存小數，很多十進位小數（例如 0.1）無法精確表示。


In [ ]:
print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)


## 2. 經典範例：Counter = ?

直覺上 `i * (1 / i)` 應該永遠等於 1，
所以 counter 應該是 100000。實際跑跑看：


In [ ]:
Max = 100000 + 1
counter = 0
for i in range(1, Max):
    x = 1 / i
    if i * x == 1:
        counter = counter + 1
print(counter)   # 不是 100000！


結果只有 86884 左右，因為 `1 / i` 產生的浮點數有誤差，
乘回去不一定剛好等於 1。

**結論：判斷相等時，不要直接用 `==` 比較浮點數。**


## 3. 如何避免使用浮點數

方法一：改用整數運算（把小數同乘 10 的次方）。

方法二：用誤差容忍值比較：`abs(a - b) < 1e-9`。


In [ ]:
# 方法一：0.1 元改用「1 角」為單位，全部變整數
price = 0.1
total_float = sum([price] * 3)
print(total_float, total_float == 0.3)

price_int = 1                      # 單位：角
total_int = price_int * 3
print(total_int / 10, total_int == 3)


In [ ]:
# 方法二：誤差容忍值
import math
a = 0.1 + 0.2
print(abs(a - 0.3) < 1e-9)
print(math.isclose(a, 0.3))


## 4. `Decimal`：十進位精確運算


In [ ]:
from decimal import Decimal

a = Decimal('0.1')
b = Decimal('0.2')
result = a + b
print(f"有使用Decimal： {a}+{b}={result}")
print(f"沒有使用Decimal： {0.1}+{0.2}={0.1+0.2}")


## 5. 注意：一定要用「字串」建立 `Decimal`

用 float 建立 `Decimal`，誤差已經跟著 float 進來了。


In [ ]:
from decimal import Decimal

a = Decimal('1.1')   # 正確使用
b = Decimal(1.1)     # 不推薦：float 的誤差被帶進來
print(a)
print(b)

x = Decimal(0.1) + Decimal(0.2)   # 錯誤示範
y = Decimal('0.1') + Decimal('0.2')
print(x)
print(y)


## 6. 設定精度：`getcontext().prec`


In [ ]:
from decimal import Decimal, getcontext

getcontext().prec = 6   # 全域精度：6 位有效數字
a = Decimal('1.123456')
b = Decimal('1.654321')
print(a * b)

getcontext().prec = 28  # 改回預設值
print(Decimal('1.123456') * Decimal('1.654321'))


## 7. `Decimal` 的代價：速度比 float 慢很多

比賽時如果題目不需要高精度，不要為了保險用 `Decimal`，會 TLE。


In [ ]:
from time import perf_counter
from decimal import Decimal

n = 1000000

start = perf_counter()
for i in range(n):
    a = 0.1
    b = 0.2
    c = a + b
print(f"float   執行時間: {perf_counter() - start:.4f} 秒")

start = perf_counter()
for i in range(n):
    a = Decimal('0.1')
    b = Decimal('0.2')
    c = a + b
print(f"Decimal 執行時間: {perf_counter() - start:.4f} 秒")


## 8. 練習題

輸入兩個小數字串 `a`、`b` 與整數 `k`，
請用 `Decimal` 精確計算 `a * b` 並輸出 `k` 位有效數字。

輸入：`a = "1.234567"`, `b = "8.765432"`, `k = 8`

期望輸出：

```text
10.821513
```


In [ ]:
# 練習作答區
a, b, k = "1.234567", "8.765432", 8


### 參考答案

In [ ]:
from decimal import Decimal, getcontext, localcontext

a, b, k = "1.234567", "8.765432", 8
with localcontext() as ctx:      # localcontext：只影響這個區塊
    ctx.prec = k
    print(Decimal(a) * Decimal(b))
